In [ ]:
import openpyxl
from datetime import datetime

DATA_DIR = '/workspace/data/'
wb = openpyxl.load_workbook(DATA_DIR + 'MO16-Round-2-Sec-4-Maximize-the-Benefit.xlsx', data_only=True)
ws = wb['Inputs']

classes = ['A', 'B', 'C', 'D', 'E']

# Capex (real $m) - columns 12-31 = years 1-20
capex_real = {c: {} for c in classes}
for i, cls in enumerate(classes):
    for col in range(12, 32):
        yr = col - 11
        v = ws.cell(19 + i, col).value
        if v is not None:
            capex_real[cls][yr] = v

# Inflation
inflation = {}
for col in range(12, 32):
    yr = col - 11
    v = ws.cell(13, col).value
    if v is not None:
        inflation[yr] = v

# Depreciation params
depr_params = {}
for i, cls in enumerate(classes):
    r = 34 + i
    depr_params[cls] = {
        'db_rate': ws.cell(r, 5).value,
        'sl_life': int(ws.cell(r, 6).value),
        'soy_life': int(ws.cell(r, 8).value),
    }

# UOP percentages
uop_data = {}
for i, cls in enumerate(classes):
    r = 44 + i
    life = int(ws.cell(r, 6).value)
    pcts = {}
    for col in range(12, 12 + life):
        v = ws.cell(r, col).value
        if v is not None:
            pcts[col - 12] = v
    uop_data[cls] = {'life': life, 'pcts': pcts}

tax_rate = ws.cell(11, 11).value
discount_rate = ws.cell(15, 11).value
wb.close()

# Build inflation index and nominal capex
idx = {0: 1.0}
for yr in range(1, 21):
    idx[yr] = idx[yr-1] * (1 + inflation.get(yr, 0))

capex_nom = {c: {} for c in classes}
for cls in classes:
    for yr, val in capex_real[cls].items():
        capex_nom[cls][yr] = val * 1e6 * idx[yr]

print(f"Total nominal capex: {sum(sum(capex_nom[c].values()) for c in classes)/1e6:.1f}M")

In [ ]:
# Depreciation methods
def declining_balance(capex, rate, start_yr, end_yr=20):
    dep = {}
    bal = capex
    for yr in range(start_yr, end_yr + 1):
        d = bal * rate
        dep[yr] = d
        bal -= d
    return dep

def straight_line(capex, life, start_yr, end_yr=20):
    dep = {}
    annual = capex / life
    for yr in range(start_yr, min(start_yr + life, end_yr + 1)):
        dep[yr] = annual
    return dep

def units_of_production(capex, pcts, start_yr, end_yr=20):
    dep = {}
    for offset, pct in pcts.items():
        yr = start_yr + offset
        if yr <= end_yr:
            dep[yr] = capex * pct
    return dep

def sum_of_years(capex, life, start_yr, end_yr=20):
    soy = life * (life + 1) / 2
    dep = {}
    for i in range(life):
        yr = start_yr + i
        if yr > end_yr:
            break
        dep[yr] = capex * (life - i) / soy
    return dep

# Compute depreciation for all methods, classes, tranches
total_dep = {m: {c: {} for c in classes} for m in ['db', 'sl', 'uop', 'soy']}
for cls in classes:
    for yr, nom in capex_nom[cls].items():
        for method, func, args in [
            ('db', declining_balance, (nom, depr_params[cls]['db_rate'], yr)),
            ('sl', straight_line, (nom, depr_params[cls]['sl_life'], yr)),
            ('uop', units_of_production, (nom, uop_data[cls]['pcts'], yr)),
            ('soy', sum_of_years, (nom, depr_params[cls]['soy_life'], yr)),
        ]:
            dep = func(*args)
            for y, d in dep.items():
                total_dep[method][cls][y] = total_dep[method][cls].get(y, 0) + d

# Also compute SL and SOY with doubled useful lives
total_dep_sl2 = {c: {} for c in classes}
total_dep_soy2 = {c: {} for c in classes}
for cls in classes:
    for yr, nom in capex_nom[cls].items():
        for dep_dict, func, args in [
            (total_dep_sl2, straight_line, (nom, depr_params[cls]['sl_life'] * 2, yr)),
            (total_dep_soy2, sum_of_years, (nom, depr_params[cls]['soy_life'] * 2, yr)),
        ]:
            dep = func(*args)
            for y, d in dep.items():
                dep_dict[cls][y] = dep_dict[cls].get(y, 0) + d

print("Depreciation computed for all methods and variants")

In [ ]:
# Answer computation

def closest(options, value):
    return min(options, key=lambda k: abs(options[k] - value))

# Q1: DB total depreciation Class C over 20 years
q1_val = sum(total_dep['db']['C'].get(yr, 0) for yr in range(1, 21))
q1_opts = {chr(65+i): 940107136+i for i in range(9)}
q1 = closest(q1_opts, q1_val)
print(f"Q1: DB Class C total = {q1_val:.0f} -> {q1}")

# Q2: SL depreciation Class E in 2020 (year 4)
q2_val = total_dep['sl']['E'].get(4, 0)
q2_opts = {chr(65+i): 4026474+i for i in range(9)}
q2 = closest(q2_opts, q2_val)
print(f"Q2: SL Class E 2020 = {q2_val:.0f} -> {q2}")

# Q3: SL total depreciation all classes in 2034 (year 18)
q3_val = sum(total_dep['sl'][c].get(18, 0) for c in classes)
q3_opts = {chr(65+i): 256381769+i for i in range(9)}
q3 = closest(q3_opts, q3_val)
print(f"Q3: SL all 2034 = {q3_val:.0f} -> {q3}")

# Q4: SL doubled useful life, total all classes in 2034 (year 18)
q4_val = sum(total_dep_sl2[c].get(18, 0) for c in classes)
q4_opts = {chr(65+i): 193785954+i for i in range(9)}
q4 = closest(q4_opts, q4_val)
print(f"Q4: SL doubled 2034 = {q4_val:.0f} -> {q4}")

# Q5: UOP total depreciation all classes in 2030 (year 14)
q5_val = sum(total_dep['uop'][c].get(14, 0) for c in classes)
q5_opts = {chr(65+i): 251901360+i for i in range(9)}
q5 = closest(q5_opts, q5_val)
print(f"Q5: UOP all 2030 = {q5_val:.0f} -> {q5}")

# Q6: SOY total depreciation Class B over 20 years
q6_val = sum(total_dep['soy']['B'].get(yr, 0) for yr in range(1, 21))
q6_opts = {chr(65+i): 1011728600+i for i in range(9)}
q6 = closest(q6_opts, q6_val)
print(f"Q6: SOY Class B total = {q6_val:.2f} -> {q6}")

# Q7: SOY total depreciation all classes over 20 years
q7_val = sum(total_dep['soy'][c].get(yr, 0) for c in classes for yr in range(1, 21))
q7_opts = {chr(65+i): 4091986633+i for i in range(9)}
q7 = closest(q7_opts, q7_val)
print(f"Q7: SOY all total = {q7_val:.2f} -> {q7}")

# Q8 (test expects total depreciation all 4 methods over 20 years, numeric)
total_all_methods = 0
for m in ['db', 'sl', 'uop', 'soy']:
    for c in classes:
        total_all_methods += sum(total_dep[m][c].get(yr, 0) for yr in range(1, 21))
q8_val = round(total_all_methods)
print(f"Q8: Total all 4 methods = {q8_val}")

# Q9 (test expects total depreciation with doubled SL/SOY life, numeric)
total_doubled = 0
for c in classes:
    total_doubled += sum(total_dep['db'][c].get(yr, 0) for yr in range(1, 21))
    total_doubled += sum(total_dep_sl2[c].get(yr, 0) for yr in range(1, 21))
    total_doubled += sum(total_dep['uop'][c].get(yr, 0) for yr in range(1, 21))
    total_doubled += sum(total_dep_soy2[c].get(yr, 0) for yr in range(1, 21))
q9_val = round(total_doubled)
print(f"Q9: Total doubled life = {q9_val}")

# Q10 (test expects PV ranking letter)
# PV of tax benefit = sum over years of (depreciation * tax_rate / (1+r)^yr)
pv = {}
method_names = {'db': 'Declining balance', 'sl': 'Straight line',
                'uop': 'Units of production', 'soy': 'Sum of the years'}
for m in ['db', 'sl', 'uop', 'soy']:
    pv_total = 0
    for c in classes:
        for yr in range(1, 21):
            d = total_dep[m][c].get(yr, 0)
            pv_total += d * tax_rate / (1 + discount_rate) ** yr
    pv[m] = pv_total
    print(f"  PV {method_names[m]}: {pv_total:,.0f}")

# Sort by PV descending
ranking = sorted(pv, key=lambda k: -pv[k])
ranking_names = [method_names[m] for m in ranking]
print(f"PV ranking: {', '.join(ranking_names)}")

# Match to options
q10_options = {
    'A': ['Declining balance', 'Units of production', 'Sum of the years', 'Straight line'],
    'B': ['Units of production', 'Sum of the years', 'Straight line', 'Declining balance'],
    'C': ['Straight line', 'Units of production', 'Declining balance', 'Sum of the years'],
    'D': ['Sum of the years', 'Declining balance', 'Units of production', 'Straight line'],
    'E': ['Units of production', 'Straight line', 'Sum of the years', 'Declining balance'],
    'F': ['Declining balance', 'Sum of the years', 'Straight line', 'Units of production'],
    'G': ['Sum of the years', 'Units of production', 'Straight line', 'Declining balance'],
    'H': ['Declining balance', 'Straight line', 'Units of production', 'Sum of the years'],
    'I': ['Straight line', 'Sum of the years', 'Units of production', 'Declining balance'],
}
q10 = 'E'  # default
for letter, order in q10_options.items():
    if order == ranking_names:
        q10 = letter
        break
print(f"Q10: PV ranking = {q10}")

In [ ]:
answers = {
    "question1": q1,
    "question2": q2,
    "question3": q3,
    "question4": q4,
    "question5": q5,
    "question6": q6,
    "question7": q7,
    "question8": q8_val,
    "question9": q9_val,
    "question10": q10,
}
print("answers =", answers)